# LABORATORIO 4

In [13]:
import pandas as pd
import numpy as np

In [14]:
# ARCHIVO DE PRUEBA
datos_sucios = """Reporte Confidencial de Ventas
Generado automaticamente por el sistema Legacy
id_transaccion;fecha_venta;cliente;ciudad;monto_usd
1001;2026-04-15; Ana Gomez ;lima;250.50
1002;2026-04-18;carlos ruiz;cusco;Error_Red
1003;2026/05/02; LUIS PEREZ;arequipa;1200.00
1003;2026/05/02; LUIS PEREZ;arequipa;1200.00

1004;2026-05-10;Maria Diaz;LIMA;No_Data
1005;2026-06-20;Jorge Soto;cusco;85.90"""
with open("ventas_legacy.csv", "w") as f:
 f.write(datos_sucios)

In [15]:
lugar = "/content/ventas_legacy.csv"

In [16]:
df_ventas = pd.read_csv(lugar, sep = ";" , skiprows=2)

In [17]:
display(df_ventas)

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,lima,250.50
1,1002,2026-04-18,carlos ruiz,cusco,Error_Red
2,1003,2026/05/02,LUIS PEREZ,arequipa,1200.00
3,1003,2026/05/02,LUIS PEREZ,arequipa,1200.00
4,1004,2026-05-10,Maria Diaz,LIMA,No_Data
5,1005,2026-06-20,Jorge Soto,cusco,85.90


2da Parte

In [18]:
# 1. Resumen técnico del DataFrame.
# Pandas asume el tipo 'object' (texto) para monto_usd porque encontró letras ("Error_Red").
print("--- 1. Resumen Técnico ---")
df_ventas.info()

# 2. Cantidad exacta de filas y columnas
print("\n--- 2. Dimensiones de la Tabla (Filas, Columnas) ---")
print(df_ventas.shape)

# 3. Verificación de valores nulos.
# Mostrará 0 nulos inicialmente porque los errores ("No_Data") aún se leen como texto válido.
print("\n--- 3. Valores Nulos Detectados Automáticamente ---")
print(df_ventas.isnull().sum())

--- 1. Resumen Técnico ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_transaccion  6 non-null      int64 
 1   fecha_venta     6 non-null      object
 2   cliente         6 non-null      object
 3   ciudad          6 non-null      object
 4   monto_usd       6 non-null      object
dtypes: int64(1), object(4)
memory usage: 372.0+ bytes

--- 2. Dimensiones de la Tabla (Filas, Columnas) ---
(6, 5)

--- 3. Valores Nulos Detectados Automáticamente ---
id_transaccion    0
fecha_venta       0
cliente           0
ciudad            0
monto_usd         0
dtype: int64


In [19]:
df_ventas['fecha_venta'] = df_ventas['fecha_venta'].str.replace('/','-')
display(df_ventas)

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,lima,250.50
1,1002,2026-04-18,carlos ruiz,cusco,Error_Red
2,1003,2026-05-02,LUIS PEREZ,arequipa,1200.00
3,1003,2026-05-02,LUIS PEREZ,arequipa,1200.00
4,1004,2026-05-10,Maria Diaz,LIMA,No_Data
5,1005,2026-06-20,Jorge Soto,cusco,85.90


In [20]:
# 1. Casteo de Fechas: Convierte la columna a formato fecha.
# errors='coerce' fuerza la conversión y vuelve NaT (Not a Time) cualquier dato ilegible.
df_ventas['fecha_venta'] = pd.to_datetime(df_ventas['fecha_venta'], errors='coerce')

# 2. Casteo Numérico: Convierte los montos a formato número (float).
# errors='coerce' transformará los textos "Error_Red" y "No_Data" en valores nulos (NaN).
df_ventas['monto_usd'] = pd.to_numeric(df_ventas['monto_usd'], errors='coerce')

# 3. Manejo de Nulos: Ahora que los errores son NaN, los rellenamos con 0 matemáticamente seguro.
df_ventas['monto_usd'] = df_ventas['monto_usd'].fillna(0)

# 4. Eliminación de Duplicados: Borra la fila de Luis Perez que estaba repetida.
df_ventas.drop_duplicates(inplace=True)

display(df_ventas)

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd
0,1001,2026-04-15,Ana Gomez,lima,250.5
1,1002,2026-04-18,carlos ruiz,cusco,0.0
2,1003,2026-05-02,LUIS PEREZ,arequipa,1200.0
4,1004,2026-05-10,Maria Diaz,LIMA,0.0
5,1005,2026-06-20,Jorge Soto,cusco,85.9


In [21]:
# 1. Limpieza de cliente: .strip() quita espacios de los bordes, .title() pone mayúscula inicial.
df_ventas['cliente'] = df_ventas['cliente'].str.strip().str.title()

# 2. Estandarización de ciudad: .upper() convierte todo a mayúsculas sostenidas.
df_ventas['ciudad'] = df_ventas['ciudad'].str.upper()

# 3. Lógica Condicional: np.where(condición, valor_si_verdadero, valor_si_falso)
df_ventas['tipo_cliente'] = np.where(df_ventas['monto_usd'] > 500, "VIP", "Regular")

display(df_ventas)

,id_transaccion,fecha_venta,cliente,ciudad,monto_usd,tipo_cliente
0,1001,2026-04-15,Ana Gomez,LIMA,250.5,Regular
1,1002,2026-04-18,Carlos Ruiz,CUSCO,0.0,Regular
2,1003,2026-05-02,Luis Perez,AREQUIPA,1200.0,VIP
4,1004,2026-05-10,Maria Diaz,LIMA,0.0,Regular
5,1005,2026-06-20,Jorge Soto,CUSCO,85.9,Regular


In [22]:
# 1. Extraer el mes: El accesor .dt funciona porque ya hicimos el casteo en la Parte 3.
df_ventas['mes_operacion'] = df_ventas['fecha_venta'].dt.month

# 2. Agrupación: Agrupamos por el mes extraído y aplicamos la suma sobre la columna de montos.
# Se usan dobles corchetes [['monto_usd']] para que el resultado siga viéndose como DataFrame.
reporte_mensual = df_ventas.groupby('mes_operacion')[['monto_usd']].sum()

# 3. Mostrar el reporte gerencial final
print("--- Reporte Regional de Ventas por Mes ---")
display(reporte_mensual)

--- Reporte Regional de Ventas por Mes ---


,monto_usd
mes_operacion,
4,250.5
5,1200.0
6,85.9
